In [ ]:
# PIPELINE GOAL: We want a dataset where each training sample represents one physical system state S:
# S(t) ​= Network-State(t) ​∪ Sensor-State(t) ​∪ Device-State(t)​
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import joblib
from sklearn.preprocessing import RobustScaler, LabelEncoder
import torch
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import pandas as pd
import numpy as np
from collections import defaultdict
import sklearn
from sklearn.model_selection import train_test_split
import os
from pandas.api.types import CategoricalDtype

cicids_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/CICIDS2017'
toniot_dir = '/content/drive/MyDrive/School Archives/Wentworth Classes/Year 2/Semester 2/Thesis 2/Datasets/TON-IoT'
CACHE_DIR = "cache"
os.makedirs(CACHE_DIR, exist_ok=True)


SENSOR_CONTINUOUS = [
    "current_temperature",
    "fridge_temperature",
    "humidity",
    "pressure",
    "temperature",
]

RATIO = [
    "Down/Up Ratio",
    "duration",
]

LOG_ROBUST = [
    "Active Max",
    "Active Mean",
    "Active Min",
    "Active Std",
    "Avg Bwd Segment Size",
    "Avg Fwd Segment Size",
    "Avg Packet Size",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Header Length",
    "Bwd IAT Max",
    "Bwd IAT Mean",
    "Bwd IAT Min",
    "Bwd IAT Std",
    "Bwd IAT Total",
    "Bwd Packet Length Max",
    "Bwd Packet Length Mean",
    "Bwd Packet Length Min",
    "Bwd Packet Length Std",
    "Bwd Packets Length Total",
    "Bwd Packets/s",
    "FC1_Read_Input_Register",
    "FC2_Read_Discrete_Value",
    "FC3_Read_Holding_Register",
    "FC4_Read_Coil",
    "Flow Bytes/s",
    "Flow IAT Max",
    "Flow IAT Min",
    "Flow IAT Std",
    "Flow Packets/s",
    "Fwd Act Data Packets",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Header Length",
    "Fwd IAT Max",
    "Fwd IAT Mean",
    "Fwd IAT Min",
    "Fwd IAT Std",
    "Fwd IAT Total",
    "Fwd Packet Length Max",
    "Fwd Packet Length Mean",
    "Fwd Packet Length Min",
    "Fwd Packet Length Std",
    "Fwd Packets Length Total",
    "Fwd Packets/s",
    "Fwd Seg Size Min",
    "Idle Max",
    "Idle Mean",
    "Idle Min",
    "Idle Std",
    "Init Bwd Win Bytes",
    "Init Fwd Win Bytes",
    "Packet Length Max",
    "Packet Length Mean",
    "Packet Length Min",
    "Packet Length Std",
    "Packet Length Variance",
    "Protocol",
    "Subflow Bwd Bytes",
    "Subflow Bwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Fwd Packets",
    "Total Backward Packets",
    "Total Fwd Packets",
    "dns_qclass",
    "dns_qtype",
    "dns_rcode",
    "dst_bytes",
    "dst_ip_bytes",
    "dst_pkts",
    "dst_port",
    "http_request_body_len",
    "http_response_body_len",
    "http_status_code",
    "latitude",
    "longitude",
    "missed_bytes",
    "src_bytes",
    "src_ip_bytes",
    "src_pkts",
    "src_port",
]

BINARY = [
    "ACK Flag Count",
    "Bwd Avg Bulk Rate",
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "CWE Flag Count",
    "ECE Flag Count",
    "FIN Flag Count",
    "Fwd Avg Bulk Rate",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "PSH Flag Count",
    "RST Flag Count",
    "SYN Flag Count",
    "URG Flag Count",
    "motion_status",
    "thermostat_status",
]

CATEGORICAL = [
    "conn_state",
    "dns_AA",
    "dns_RA",
    "dns_RD",
    "dns_query",
    "dns_rejected",
    "door_state",
    "http_method",
    "http_orig_mime_types",
    "http_resp_mime_types",
    "http_trans_depth",
    "http_uri",
    "http_user_agent",
    "http_version",
    "light_status",
    "proto",
    "service",
    "sphone_signal",
    "ssl_cipher",
    "ssl_established",
    "ssl_issuer",
    "ssl_resumed",
    "ssl_subject",
    "ssl_version",
    "temp_condition",
    "weird_addl",
    "weird_name",
    "weird_notice",
]


def load_raw_file(path):
  if path.endswith('.parquet'):
    return pd.read_parquet(path)
  if path.endswith('.csv'):
    return pd.read_csv(path)

  raise ValueError(f"Unsupported file type: {path}")


def unify_target(df):
  for col in ["type", "Label", "label"]:
    if col in df.columns:
      df["target"] = df[col]
      df.drop(columns=[col], inplace=True)
      df["target"] = df["target"].astype(str).str.strip()
      df["target"] = df["target"].str.lower()
      break

  if "target" not in df.columns:
    raise ValueError("No attack label found")

  return df


def preprocess_dataframe(df):
  df = unify_target(df)
  try:
    df = df.apply(pd.to_numeric, errors='ignore')
  except ValueError:
    raise ValueError("Invalid data type")
  df.replace([np.inf, -np.inf], np.nan, inplace=True)

  num_cols = df.select_dtypes(include=[np.number]).columns
  cat_cols = df.select_dtypes(include=["category", "object"]).columns

  df[num_cols] = df[num_cols].fillna(0)

  for c in cat_cols:
      df[c] = df[c].astype(str).fillna("unknown")

  return df


# enterprise network/N(t) - network features
def load_all_cicids():
  df_all = None

  for file in os.listdir(cicids_dir):
    if file.endswith('.parquet'):
      path = os.path.join(cicids_dir, file)
      df = load_raw_file(path)
      df = preprocess_dataframe(df)

      if df_all is None:
        df_all = df

      else:
        df_all = pd.concat([df_all, df], ignore_index=True)

  return df_all


# IoT network traffic/P(t) - physical sensor features
def load_ton_network():
  df_all = None
  for root, _, files in os.walk(toniot_dir):
    for f in files:
      if f.endswith('network.csv'):
        path = os.path.join(root, f)
        df = load_raw_file(path)
        df = preprocess_dataframe(df)

        if df_all is None:
          df_all = df

        else:
          df_all = pd.concat([df_all, df], ignore_index=True)

  return df_all


# physical sensor telemetry/D(t) - device/logical state
def load_ton_sensor():
  df_all = None

  for root, _, files in os.walk(toniot_dir):
    for f in files:
      if f.endswith('.csv') and "network" not in f:
        path = os.path.join(root, f)
        df = load_raw_file(path)
        df = preprocess_dataframe(df)

        if df_all is None:
          df_all = df

        else:
          df_all = pd.concat([df_all, df], ignore_index=True)

  return df_all


def safe_log_transform(df, cols):
  df = df.copy()
  for c in cols:
      col = df[c].values
      if np.all(np.isnan(col)):
        continue

      clip_val = np.nanpercentile(col, 99.9)

      col = np.clip(col, 0, clip_val)
      df[c] = np.log1p(col)

  return df


def select_existing(df, cols):
  return df[[c for c in cols if c in df.columns]]


def build_timestamp(df):
  if "date" in df.columns and "time" in df.columns:
    # TON-IoT absolute time
    ts = pd.to_datetime(df["date"].astype(str) + " " + df["time"].astype(str), format="%Y-%m-%d %H:%M:%S", errors="coerce")
    ts = ts.fillna(method="ffill")

  elif "duration" in df.columns:
    delta_t = pd.to_numeric(df["duration"], errors="coerce").fillna(0)
    ts = pd.to_timedelta(delta_t, unit="s").cumsum()

  # CICIDS synthetic time
  elif "Flow Duration" in df.columns:
    delta_t = pd.to_numeric(df["Flow Duration"], errors="coerce").fillna(0)
    ts = pd.to_timedelta(delta_t, unit="s").cumsum()

  elif "Flow IAT Mean" in df.columns:
    delta_t = pd.to_numeric(df["Flow IAT Mean"], errors="coerce").fillna(0)
    ts = pd.to_timedelta(delta_t, unit="s").cumsum()

  else:
    ts = pd.to_timedelta(np.arange(len(df)), unit="s")

  ts = ts.fillna(method="ffill").fillna(method="bfill")

  if pd.api.types.is_timedelta64_dtype(ts):
    ts = pd.Timestamp('1970-01-01') + ts

  return pd.DataFrame({"__ts__": ts})


class SlidingWindowDataset(torch.utils.data.Dataset):

  def __init__(self, X, y, window, label_encoder):
    X_sorted = X.sort_values("__ts__")
    y_sorted = y.loc[X_sorted.index]

    X_num = X_sorted.drop(columns=["__ts__"])
    X_num = X_num.apply(pd.to_numeric, errors="coerce").fillna(0)

    self.X = X_num.values.astype(np.float32)
    self.y = label_encoder.transform(y_sorted.values).astype(np.int64)
    self.window = window

  def __len__(self):
    return len(self.X) - self.window + 1

  def __getitem__(self, idx):
    x = torch.from_numpy(self.X[idx:idx+self.window])
    y = torch.tensor(self.y[idx+self.window-1])
    return x, y


def build_fused_features_raw(df):
  df = df.copy()

  drop_cols = ["src_ip", "dst_ip", "date", "time"]

  for col in drop_cols:
    if col in df.columns:
      df.drop(columns=[col], inplace=True)

  ts = df["__ts__"]
  X_net = select_existing(df, LOG_ROBUST)
  X_sens = select_existing(df, SENSOR_CONTINUOUS)
  X_bin = select_existing(df, BINARY)
  X_cat = select_existing(df, CATEGORICAL)
  X_ratio = select_existing(df, RATIO)

  X = pd.concat([X_net, X_sens, X_bin, X_cat, X_ratio], axis=1)

  X["__ts__"] = ts.values
  X["target"] = df["target"].values

  return X


def transform_fused_features(df):
  df = df.copy()
  X_net = select_existing(df, LOG_ROBUST)
  X_sens = select_existing(df, SENSOR_CONTINUOUS)
  X_bin = select_existing(df, BINARY)
  X_cat = select_existing(df, CATEGORICAL)
  X_ratio = select_existing(df, RATIO)

  X_net = transform_network(X_net)
  X_sens = transform_sensor(X_sens)
  X_ratio = transform_ratio(X_ratio)
  X_bin = select_binary(X_bin)
  X_cat = encode_categorical(X_cat)

  X = pd.concat([X_net, X_sens, X_bin, X_cat, X_ratio], axis=1)

  return X


def transform_network(df):
  df = df.copy()
  if not os.path.exists("profiles/network_log_robust_scaler.pkl"):
      build_network_profile(df)
  bundle = joblib.load("profiles/network_log_robust_scaler.pkl")
  scaler = bundle["scaler"]
  fit_cols = bundle["columns"]

  for col in fit_cols:
      if col not in df.columns:
          df[col] = 0

  df_fit = df[fit_cols].copy()

  df_fit = safe_log_transform(df_fit, fit_cols)

  df_fit = scaler.transform(df_fit)

  df[fit_cols] = df_fit

  return df


def transform_sensor(df):
  df = df.copy()
  for col in SENSOR_CONTINUOUS:
      if col not in df:
          continue

      if "temp" in col:
          df[col] = (df[col] - 20) / 15
      elif "humidity" in col:
          df[col] = df[col] / 100
      elif "pressure" in col:
          df[col] = (df[col] - 1013) / 200

  return df


def transform_ratio(df):
  df = df.copy()
  for col in RATIO:
      if col not in df:
          continue

      df[col] = np.log1p(df[col])

  return df


def select_binary(df):
  df = df.copy()
  return select_existing(df, BINARY).astype(float)


def encode_categorical(df, profile_path="profiles/categorical.pkl"):
  df = df.copy()

  cats = joblib.load(profile_path)

  for col, values in cats.items():
    if col not in df.columns:
      df[col] = "unknown"

    all_categories = values + ["unknown"]
    # Convert to categorical with predefined categories. Values not in `all_categories` will become NaN.
    df[col] = pd.Categorical(df[col], categories=all_categories)
    # Fill NaN values (which are the original values not in `all_categories`) with 'unknown'.
    df[col] = df[col].fillna("unknown")
    # Convert to numerical codes.
    df[col] = df[col].cat.codes

  return df


def build_network_profile(train_df, profile_path="profiles/network_log_robust_scaler.pkl"):
    os.makedirs("profiles", exist_ok=True)

    df_all = select_existing(train_df, LOG_ROBUST)

    df_all = safe_log_transform(df_all, df_all.columns)

    scaler = RobustScaler().fit(df_all)

    joblib.dump(
        {
            "scaler": scaler,
            "columns": df_all.columns.tolist()
        }, profile_path)


def build_categorical_profile(train_df, profile_path="profiles/categorical.pkl"):
    os.makedirs("profiles", exist_ok=True)

    cats= {}

    for col in CATEGORICAL:
      if col in train_df.columns:
        cats[col] = sorted(train_df[col].astype(str).unique().tolist())

    joblib.dump(cats, profile_path)


def get_label_encoder(y_train):
    le = LabelEncoder()
    le.fit(y_train)
    return le


# align P(t) + D(t) modality
def align_modalities(net_df, sensor_df):

  # Ensure __ts__ columns are datetime64[ns] for merging
  if pd.api.types.is_timedelta64_dtype(net_df["__ts__"]):
    net_df["__ts__"] = pd.Timestamp('1970-01-01') + net_df["__ts__"]
  if pd.api.types.is_timedelta64_dtype(sensor_df["__ts__"]):
    sensor_df["__ts__"] = pd.Timestamp('1970-01-01') + sensor_df["__ts__"]

  net_df = net_df.dropna(subset=["__ts__"])
  sensor_df = sensor_df.dropna(subset=["__ts__"])

  net_df = net_df.sort_values(by="__ts__")
  sensor_df = sensor_df.sort_values(by="__ts__")

  aligned  = pd.merge_asof(
      net_df,
      sensor_df,
      on="__ts__",
      direction="nearest",
      tolerance=pd.Timedelta(seconds=30)
  )

  return aligned


def get_torch_loaders_sequence(train_dataset, val_dataset, test_dataset, le, batch_size):

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=2, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=2, pin_memory=True)

  n_features = train_dataset.X.shape[1]
  n_classes = len(le.classes_)

  return train_loader, val_loader, test_loader, n_features, n_classes


def get_torch_loaders_tabular_static(X_train, y_train, X_val, y_val, X_test, y_test, le, batch_size):
  def to_torch(X, y, le):
    y_enc = le.transform(y)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)
    X_t = torch.tensor(X.values.astype(np.float32))
    y_t = torch.tensor(y_enc, dtype=torch.long)
    return X_t, y_t

  Xtr, ytr = to_torch(X_train, y_train, le)
  Xva, yva = to_torch(X_val, y_val, le)
  Xte, yte = to_torch(X_test, y_test, le)

  class_counts = torch.bincount(ytr)
  class_weights = 1.0 / torch.clamp(class_counts.float(), min=1)

  sample_weights = class_weights[ytr]

  sampler = WeightedRandomSampler(
      weights=sample_weights,
      num_samples=len(sample_weights),
      replacement=True
  )

  train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, sampler=sampler, shuffle=False, num_workers=2, pin_memory=True)
  val_loader = DataLoader(TensorDataset(Xva, yva), batch_size=batch_size, num_workers=2, pin_memory=True)
  test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=batch_size, num_workers=2, pin_memory=True)

  n_features = Xtr.shape[1]
  n_classes = len(le.classes_)

  return train_loader, val_loader, test_loader, n_features, n_classes

"""
Question: Why are static models trained on random splits while sequence models
          are trained on temporal splits?
Answer: Static models assume independence while sequence models assume temporal
        dependencies. Therefore, splitting strategy differs by hypothesis class.
"""

def get_full_pipeline(mode: int, batch_size=None):
  # load raw data
  cache_file = os.path.join(CACHE_DIR, f"tabular_raw_mode_{mode}.parquet")
  splits_dir = os.path.join(CACHE_DIR, f"tabular_splits_static_mode_{mode}")
  encoder_scaler_dir = os.path.join(CACHE_DIR, f"encoder_scaler_static_mode_{mode}")

  if os.path.exists(cache_file):
    print(f"Loading cached data from {cache_file}...")
    df_all = pd.read_parquet(cache_file)

  else:
    print("Building dataset from raw files...")

    if mode == 0:
      df_cicids = load_all_cicids()
      df_ton_net = load_ton_network()
      df_cicids = pd.concat([df_cicids, build_timestamp(df_cicids)], axis=1)
      df_ton_net = pd.concat([df_ton_net, build_timestamp(df_ton_net)], axis=1)
      df_all = pd.concat([df_cicids, df_ton_net], ignore_index=True)

    elif mode == 1:
      df_ton_sens = load_ton_sensor()
      df_ton_sens = pd.concat([df_ton_sens, build_timestamp(df_ton_sens)], axis=1)
      df_all = df_ton_sens

    elif mode == 2:
      df_cicids = load_all_cicids()
      df_ton_net = load_ton_network()
      df_ton_sens = load_ton_sensor()
      df_cicids = pd.concat([df_cicids, build_timestamp(df_cicids)], axis=1)
      df_ton_net = pd.concat([df_ton_net, build_timestamp(df_ton_net)], axis=1)
      df_ton_sens = pd.concat([df_ton_sens, build_timestamp(df_ton_sens)], axis=1)
      aligned_ton = align_modalities(df_ton_net, df_ton_sens)
      df_all = pd.concat([df_cicids, aligned_ton], ignore_index=True)

    df_all = df_all.dropna(subset=["target"])

    min_samples = 20

    counts = df_all["target"].value_counts()

    valid_labels = counts[counts >= min_samples].index

    df_all = df_all[df_all["target"].isin(valid_labels)]

    df_all.to_parquet(cache_file)

  if os.path.exists(splits_dir):
    print(f"Loading cached splits from {splits_dir}...")
    X_train = pd.read_parquet(os.path.join(splits_dir, "X_train.parquet"))
    y_train = pd.read_parquet(os.path.join(splits_dir, "y_train.parquet"))
    X_val = pd.read_parquet(os.path.join(splits_dir, "X_val.parquet"))
    y_val = pd.read_parquet(os.path.join(splits_dir, "y_val.parquet"))
    X_test = pd.read_parquet(os.path.join(splits_dir, "X_test.parquet"))
    y_test = pd.read_parquet(os.path.join(splits_dir, "y_test.parquet"))

  else:
    # stratify
    X = df_all.drop(columns=["target"], errors="ignore")
    y = df_all["target"]

    X_tr, X_temp, y_tr, y_temp = train_test_split(
      X, y, test_size=0.30, stratify=y, random_state=42
    )

    X_va, X_te, y_va, y_te = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
    )

    train = pd.concat([X_tr, y_tr], axis=1)
    val = pd.concat([X_va, y_va], axis=1)
    test = pd.concat([X_te, y_te], axis=1)

    seen = set(train["target"].unique())
    val = val[val["target"].isin(seen)].copy()
    test = test[test["target"].isin(seen)].copy()

    # prep splits for transformation
    y_train = train["target"].copy()
    X_train = build_fused_features_raw(train)

    if not os.path.exists("profiles/categorical.pkl"):
      build_categorical_profile(X_train)

    y_val = val["target"].copy()
    X_val = build_fused_features_raw(val)
    y_test = test["target"].copy()
    X_test = build_fused_features_raw(test)

    # fit scaler only on train split
    if not os.path.exists("profiles/network_log_robust_scaler.pkl"):
      train_tmp = X_train.copy()
      train_tmp["target"] = y_train
      build_network_profile(train_tmp)

    # transform per X (preserve labels for y)
    X_train = transform_fused_features(X_train)
    X_val = transform_fused_features(X_val)
    X_test = transform_fused_features(X_test)

    X_train = X_train.drop(columns=["target"], errors="ignore").drop(columns=["__ts__"], errors="ignore")
    X_val = X_val.drop(columns=["target"], errors="ignore").drop(columns=["__ts__"], errors="ignore")
    X_test = X_test.drop(columns=["target"], errors="ignore").drop(columns=["__ts__"], errors="ignore")

    os.makedirs(splits_dir, exist_ok=True)

    X_train.to_parquet(os.path.join(splits_dir, "X_train.parquet"))
    y_train.to_parquet(os.path.join(splits_dir, "y_train.parquet"))

    X_val.to_parquet(os.path.join(splits_dir, "X_val.parquet"))
    y_val.to_parquet(os.path.join(splits_dir, "y_val.parquet"))

    X_test.to_parquet(os.path.join(splits_dir, "X_test.parquet"))
    y_test.to_parquet(os.path.join(splits_dir, "y_test.parquet"))

  if os.path.exists(encoder_scaler_dir):
    print(f"Loading cached encoder and scaler from {encoder_scaler_dir}...")
    scaler = joblib.load(os.path.join(encoder_scaler_dir, "scaler.pkl"))
    le = joblib.load(os.path.join(encoder_scaler_dir, "le.pkl"))

  else:
    print("Fitting encoder and scaler...")
    scaler = None
    le = get_label_encoder(y_train)
    joblib.dump(scaler, os.path.join(encoder_scaler_dir, "scaler.pkl"))
    joblib.dump(le, os.path.join(encoder_scaler_dir, "le.pkl"))

  print("Building loaders...")
  # tabular static loader for each split
  loaders = get_torch_loaders_tabular_static(X_train, y_train, X_val, y_val, X_test, y_test, le, batch_size)

  return loaders, scaler, le


def get_full_sequence_pipeline(mode: int, window=None, batch_size=None):
  # load raw and fit timestamps first
  cache_file = os.path.join(CACHE_DIR, f"tabular_raw_mode_{mode}.parquet")
  splits_dir = os.path.join(CACHE_DIR, f"tabular_splits_sequential_mode_{mode}_{window}")
  encoder_dir = os.path.join(CACHE_DIR, f"encoder_sequential_mode_{mode}_{window}")

  if os.path.exists(cache_file):
    print(f"Loading cached data from {cache_file}...")
    df_all = pd.read_parquet(cache_file)

  else:
    print("Building dataset from raw files...")

    if mode == 0:
      df_cicids = load_all_cicids()
      df_ton_net = load_ton_network()
      df_cicids = pd.concat([df_cicids, build_timestamp(df_cicids)], axis=1)
      df_ton_net = pd.concat([df_ton_net, build_timestamp(df_ton_net)], axis=1)
      df_all = pd.concat([df_cicids, df_ton_net], ignore_index=True)

    elif mode == 1:
      df_ton_sens = load_ton_sensor()
      df_ton_sens = pd.concat([df_ton_sens, build_timestamp(df_ton_sens)], axis=1)
      df_all = df_ton_sens

    elif mode == 2:
      df_cicids = load_all_cicids()
      df_ton_net = load_ton_network()
      df_ton_sens = load_ton_sensor()
      df_cicids = pd.concat([df_cicids, build_timestamp(df_cicids)], axis=1)
      df_ton_net = pd.concat([df_ton_net, build_timestamp(df_ton_net)], axis=1)
      df_ton_sens = pd.concat([df_ton_sens, build_timestamp(df_ton_sens)], axis=1)
      aligned_ton = align_modalities(df_ton_net, df_ton_sens)
      df_all = pd.concat([df_cicids, aligned_ton], ignore_index=True)

    df_all = df_all.dropna(subset=["target"])

    min_samples = 20

    counts = df_all["target"].value_counts()

    valid_labels = counts[counts >= min_samples].index

    df_all = df_all[df_all["target"].isin(valid_labels)]

    df_all.to_parquet(cache_file)

  if os.path.exists(splits_dir):
    print(f"Loading cached splits from {splits_dir}...")
    train_dataset = pd.read_parquet(os.path.join(splits_dir, "train_dataset.parquet"))
    val_dataset = pd.read_parquet(os.path.join(splits_dir, "val_dataset.parquet"))
    test_dataset = pd.read_parquet(os.path.join(splits_dir, "test_dataset.parquet"))

  else:
    print("Building splits...")
    # sort by __ts__
    df_all = df_all.sort_values(by="__ts__")
    n = len(df_all)

    # sequential (non-stratified)
    train_end = int(0.7 * n)
    val_end = int(0.85 * n)

    train = df_all.iloc[:train_end]
    val = df_all.iloc[train_end:val_end]
    test = df_all.iloc[val_end:]

    seen = set(train["target"].unique())
    val = val[val["target"].isin(seen)].copy()
    test = test[test["target"].isin(seen)].copy()

    # prep splits for transformation
    y_train = train["target"].copy()
    X_train = build_fused_features_raw(train)

    if not os.path.exists("profiles/categorical.pkl"):
      build_categorical_profile(X_train)

    ts_train = X_train["__ts__"].copy()
    y_val = val["target"].copy()
    X_val = build_fused_features_raw(val)
    ts_val = X_val["__ts__"].copy()
    y_test = test["target"].copy()
    X_test = build_fused_features_raw(test)
    ts_test = X_test["__ts__"].copy()

    # fit scaler only on train split
    if not os.path.exists("profiles/network_log_robust_scaler.pkl"):
      train_tmp = X_train.copy()
      train_tmp["target"] = y_train
      build_network_profile(train_tmp)

    # transform per X (preserve labels for y)
    X_train = transform_fused_features(X_train)
    X_train["__ts__"] = ts_train.values
    X_train = X_train.apply(pd.to_numeric, errors="coerce").fillna(0)

    X_val = transform_fused_features(X_val)
    X_val["__ts__"] = ts_val.values
    X_val = X_val.apply(pd.to_numeric, errors="coerce").fillna(0)

    X_test = transform_fused_features(X_test)
    X_test["__ts__"] = ts_test.values
    X_test = X_test.apply(pd.to_numeric, errors="coerce").fillna(0)

    # extract X and y from each split
    X_train = X_train.drop(columns=["target"], errors="ignore")
    X_val = X_val.drop(columns=["target"], errors="ignore")
    X_test = X_test.drop(columns=["target"], errors="ignore")

    if os.path.exists(encoder_dir):
      print(f"Loading cached encoder from {encoder_dir}...")
      le = joblib.load(os.path.join(encoder_dir, "le.pkl"))

    else:
      print("Fitting encoder...")
      le = get_label_encoder(y_train)
      os.makedirs(encoder_dir, exist_ok=True)
      joblib.dump(le, os.path.join(encoder_dir, "le.pkl"))

    # apply windows per split
    if window is not None:
      train_dataset = SlidingWindowDataset(X_train, y_train, window, le)
      val_dataset = SlidingWindowDataset(X_val, y_val, window, le)
      test_dataset = SlidingWindowDataset(X_test, y_test, window, le)

    else:
      raise ValueError("Windows must be provided for sequence models")

  scaler = None

  print("Building loaders...")
  # sequential loader for each split
  if window is not None:
    loaders = get_torch_loaders_sequence(
      train_dataset, val_dataset, test_dataset, le, batch_size
    )


  return loaders, scaler, le